# 🧠 Stage 1: Benchmark Ingestion & Verification
**Project:** Reduction Ladder for Code & Multi-Arm Mitigation  
**Organization:** Orange Innovation Labs  
**Authors:** Omar Abdelhamid, Nour Walid  
**Supervisor:** Dr. Ghada  

---

### 🎯 Objectives:
1. Ingest **L0 to L5** benchmark datasets directly from Hugging Face (`openai/openai_humaneval` and `evoeval/EvoEval_*`).
2. Standardize all problem schemas into a unified JSONL format.
3. Execute sandbox verification on canonical ground-truth solutions.
4. Inspect samples across all 6 ladder levels.

In [ ]:
import os
import sys
import json
import pandas as pd

# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath(".."))

from src.data_pipeline.loader import LADDER_BENCHMARKS, download_level, save_tasks_to_jsonl, fetch_all_ladder_levels
from src.data_pipeline.verifier import verify_dataset_tasks, run_sandbox_execution

print("✅ Modules imported successfully!")

## 1. Overview of the Reduction Ladder Levels
Let's inspect the target benchmarks and their Hugging Face sources.

In [ ]:
ladder_df = pd.DataFrame.from_dict(LADDER_BENCHMARKS, orient="index")
ladder_df.index.name = "Level"
ladder_df

## 2. Ingest and Normalize All Ladder Levels (L0 to L5)

In [ ]:
output_directory = "../data/ladder"
all_ladder_data = fetch_all_ladder_levels(output_dir=output_directory)

print(f"\n🎉 All {len(all_ladder_data)} levels loaded and saved to '{output_directory}'!")

## 3. Ground-Truth Sandbox Verification Suite
Run unit tests on all canonical solutions to ensure 100% test-suite correctness.

In [ ]:
verification_results = {}
for level_key, tasks in all_ladder_data.items():
    print(f"\n==================== Verifying {level_key} ====================")
    res = verify_dataset_tasks(tasks)
    verification_results[level_key] = res

summary_df = pd.DataFrame([
    {"Level": k, "Total Tasks": v["total"], "Passed": v["passed"], "Pass Rate (%)": v["pass_rate"]}
    for k, v in verification_results.items()
])
summary_df

## 4. Visual Inspection: Comparing Problem Transformations Across Levels
Let's look at the first problem (`HumanEval/0`) transformed across L0, L1, L2, L3, L4, and L5.

In [ ]:
for level in ["L0", "L1", "L2", "L3", "L4", "L5"]:
    task = all_ladder_data[level][0]
    print(f"\n{'='*30} [{level}]: {task['benchmark']} {'='*30}")
    print(f"Task ID: {task['task_id']}")
    print("Prompt:")
    print(task["prompt"][:300] + "...\n")